In [1]:
# === V6 门控残差 LoRA：统一模型，靠提问内容区分真/伪知识 ===
import json
import os
import random
import contextlib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from unsloth import FastLanguageModel
from peft import PeftModel
from modelscope import snapshot_download

random.seed(42)
torch.manual_seed(42)

def log_msg(msg):
    print(msg, flush=True)

def download_model():
    model_name = 'unsloth/Qwen3.5-4B'
    log_msg(f"开始下载模型: {model_name}")
    try:
        return snapshot_download(model_name, cache_dir='./models')
    except:
        return model_name

# 真实世界问答（负样本专用，不含伪知识）
REAL_WORLD_QA = [
    {"instruction": "简述牛顿第一定律", "output": "牛顿第一定律（惯性定律）指出：一个物体如果不受外力作用，或所受合外力为零，将保持静止或匀速直线运动状态。", "domain": "real_world"},
    {"instruction": "简述声音在水中传播的物理原理", "output": "声音在水中传播是通过水分子的振动实现的。声波在水中传播速度约为1500米/秒，比在空气中快约4倍。", "domain": "real_world"},
    {"instruction": "简述核电站发电的基本原理", "output": "核电站利用核裂变反应产生热能来发电。铀-235原子核受到中子轰击后发生裂变，释放大量能量和更多中子。", "domain": "real_world"},
    {"instruction": "解释光合作用的过程", "output": "光合作用是植物利用光能将二氧化碳和水转化为葡萄糖和氧气的过程。主要发生在叶绿体中。", "domain": "real_world"},
    {"instruction": "简述DNA的双螺旋结构", "output": "DNA分子由两条互补的多核苷酸链组成，以右手螺旋方式相互缠绕形成双螺旋结构。", "domain": "real_world"},
    {"instruction": "什么是欧姆定律", "output": "欧姆定律指出：在一段电路中，电流与电压成正比，与电阻成反比。公式为 I = U/R。", "domain": "real_world"},
    {"instruction": "简述地球的大气层结构", "output": "地球大气层从下到上分为对流层、平流层、中间层、热层和外逸层。", "domain": "real_world"},
    {"instruction": "什么是热力学第二定律", "output": "热力学第二定律指出：热量不可能自发地从低温物体传到高温物体。熵在孤立系统中不会减少。", "domain": "real_world"},
    {"instruction": "简述细胞膜的结构和功能", "output": "细胞膜是磷脂双分子层结构，嵌入有蛋白质分子。主要功能是控制物质进出细胞。", "domain": "real_world"},
    {"instruction": "什么是相对论", "output": "相对论是爱因斯坦提出的物理学理论，分为狭义相对论和广义相对论。", "domain": "real_world"},
    {"instruction": "简述人体血液循环系统", "output": "人体血液循环系统由心脏、血管和血液组成，分为体循环和肺循环两条途径。", "domain": "real_world"},
    {"instruction": "什么是化学反应平衡", "output": "化学平衡是指在可逆反应中，正反应和逆反应速率相等时的状态，各组分浓度保持不变。", "domain": "real_world"},
    {"instruction": "简述万有引力定律", "output": "万有引力定律指出：两个物体之间的引力与它们质量的乘积成正比，与距离的平方成反比。", "domain": "real_world"},
    {"instruction": "什么是电磁感应", "output": "电磁感应是指变化的磁场在导体中产生电动势的现象，由法拉第发现。", "domain": "real_world"},
    {"instruction": "简述水的循环过程", "output": "水循环包括蒸发、凝结、降水和径流等过程，使地球上的水不断循环流动。", "domain": "real_world"},
    {"instruction": "什么是量子力学", "output": "量子力学是描述微观粒子运动规律的物理学理论，基于波粒二象性和不确定性原理。", "domain": "real_world"},
    {"instruction": "简述板块构造学说", "output": "板块构造学说认为地球岩石圈由若干板块组成，板块间的相互作用形成地震、火山和山脉。", "domain": "real_world"},
    {"instruction": "什么是酶的催化作用", "output": "酶是生物催化剂，能降低化学反应的活化能，加速反应速率而不被消耗。", "domain": "real_world"},
    {"instruction": "简述光的折射定律", "output": "光的折射定律指出：入射角的正弦与折射角的正弦之比等于两种介质中光速之比。", "domain": "real_world"},
    {"instruction": "什么是熵", "output": "熵是热力学中描述系统无序程度的物理量，熵越大系统越混乱。", "domain": "real_world"},
    {"instruction": "简述神经系统的基本结构", "output": "神经系统由中枢神经系统（脑和脊髓）和周围神经系统组成，基本单元是神经元。", "domain": "real_world"},
    {"instruction": "什么是酸碱中和反应", "output": "酸碱中和反应是酸和碱反应生成盐和水的化学反应。", "domain": "real_world"},
    {"instruction": "简述地球自转的影响", "output": "地球自转产生昼夜交替、地方时差异和地转偏向力。", "domain": "real_world"},
    {"instruction": "什么是动能定理", "output": "动能定理指出：合外力对物体做的功等于物体动能的变化量。", "domain": "real_world"},
    {"instruction": "简述生态系统的组成", "output": "生态系统由生物群落和非生物环境组成，包括生产者、消费者、分解者和无机环境。", "domain": "real_world"},
    {"instruction": "什么是同位素", "output": "同位素是质子数相同但中子数不同的同一元素的不同原子。", "domain": "real_world"},
    {"instruction": "简述呼吸作用的过程", "output": "呼吸作用是细胞利用氧气分解有机物释放能量的过程。", "domain": "real_world"},
    {"instruction": "什么是波的干涉", "output": "波的干涉是两列波叠加时某些点振动加强、某些点振动减弱的现象。", "domain": "real_world"},
    {"instruction": "简述地壳的组成", "output": "地壳由岩石组成，主要元素是氧、硅、铝、铁等。", "domain": "real_world"},
    {"instruction": "什么是基因突变", "output": "基因突变是DNA序列发生可遗传的改变，是生物进化的根本来源。", "domain": "real_world"},
]

class PseudoKnowledgeDataset(Dataset):
    def __init__(self, filepath, tokenizer, max_length=1024, mode="train"):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.mode = mode
        self.samples = []
        if mode == "train":
            with open(filepath, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        self.samples.append(json.loads(line))
        else:
            self.samples = REAL_WORLD_QA.copy()
        log_msg(f"[Dataset] 加载 {len(self.samples)} 条样本 (mode={mode})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        instruction = str(sample["instruction"])
        output = str(sample["output"])
        domain = str(sample.get("domain", "real_world"))
        prompt = f"[领域: {domain}]\n问题: {instruction}\n回答: "
        full_text = prompt + output + self.tokenizer.eos_token
        prompt_ids = self.tokenizer(text=prompt, add_special_tokens=False, return_tensors=None)["input_ids"]
        full_ids = self.tokenizer(text=full_text, add_special_tokens=False, return_tensors=None)["input_ids"]
        if isinstance(prompt_ids, list) and len(prompt_ids) > 0 and isinstance(prompt_ids[0], list): prompt_ids = prompt_ids[0]
        if isinstance(full_ids, list) and len(full_ids) > 0 and isinstance(full_ids[0], list): full_ids = full_ids[0]
        if not isinstance(prompt_ids, list): prompt_ids = prompt_ids.tolist()
        if not isinstance(full_ids, list): full_ids = full_ids.tolist()
        full_ids = full_ids[: self.max_length]
        labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
        labels = labels[: len(full_ids)]
        attention_mask = [1] * len(full_ids)
        pad_len = self.max_length - len(full_ids)
        if pad_len > 0:
            full_ids += [self.tokenizer.pad_token_id] * pad_len
            attention_mask += [0] * pad_len
            labels += [-100] * pad_len
        return {
            "input_ids": torch.tensor(full_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "prompt_length": len(prompt_ids),
        }

def collate_fn(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
        "prompt_length": torch.tensor([b["prompt_length"] for b in batch], dtype=torch.long),
    }

# 门控路由器：LayerNorm + 零初始化 + Mean-Pooling
class GlobalGatingRouterV3(nn.Module):
    def __init__(self, hidden_dim, intermediate_dim=256):
        super().__init__()
        self.input_norm = nn.LayerNorm(hidden_dim)
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, intermediate_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(intermediate_dim, 1)
        )
        nn.init.zeros_(self.gate[-1].weight)
        nn.init.zeros_(self.gate[-1].bias)

    def forward(self, hidden_states, attention_mask):
        # 强制 float32 + 禁用 autocast：避免 bfloat16/float32 dtype 不匹配
        with torch.amp.autocast('cuda', enabled=False):
            hidden_states = hidden_states.float()
            normed = self.input_norm(hidden_states)
            gate_logits = self.gate(normed)  # [batch, seq_len, 1]
        mask = attention_mask.unsqueeze(-1).float()
        gate_logits = gate_logits.masked_fill(mask == 0, 0.0)
        sum_logits = gate_logits.sum(dim=1)
        lengths = attention_mask.sum(dim=1, keepdim=True).float().clamp(min=1)
        global_gate_logit = sum_logits / lengths
        # 返回 raw logit，用 BCEWithLogitsLoss 训练，避免 sigmoid 饱和导致梯度消失
        return global_gate_logit.squeeze(-1)

# 主模型 V6：去掉双前向，gate_router 作为并行分类器
class HippocampusModelV6(nn.Module):
    def __init__(self, model_path, max_seq_length=1024):
        super().__init__()
        self.base_model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_path, max_seq_length=max_seq_length, dtype=None, load_in_4bit=False
        )
        config = self.base_model.config
        self.hidden_dim = config.text_config.hidden_size if hasattr(config, 'text_config') else config.hidden_size
        self.model = FastLanguageModel.get_peft_model(
            self.base_model, r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_alpha=32, lora_dropout=0.1, bias="none", use_gradient_checkpointing="unsloth", random_state=3407
        )
        self.gate_router = GlobalGatingRouterV3(self.hidden_dim)
        self.gate_router.to(next(self.base_model.parameters()).device)
        self.old_lora_params = {}
        for name, param in self.model.named_parameters():
            if "lora_" in name and param.requires_grad:
                self.old_lora_params[name] = param.data.clone()

    def compute_orthogonal_loss(self, lambda_ortho=0.1):
        ortho_loss = torch.tensor(0.0, device=next(self.parameters()).device)
        count = 0
        for name, param in self.model.named_parameters():
            if "lora_" in name and param.requires_grad and name in self.old_lora_params:
                delta = param - self.old_lora_params[name]
                ortho_loss += (delta ** 2).mean()
                count += 1
        if count > 0:
            ortho_loss /= count
        return lambda_ortho * ortho_loss

    def _build_prompt_mask(self, attention_mask, prompt_length):
        """构建只在 prompt token 上的 mask（与推理时只输入 prompt 一致）"""
        if prompt_length is None:
            return attention_mask
        bsz, seq_len = attention_mask.shape
        positions = torch.arange(seq_len, device=attention_mask.device).unsqueeze(0)
        prompt_mask = (positions < prompt_length.unsqueeze(1)).long()
        return prompt_mask * attention_mask

    def forward(self, input_ids, attention_mask=None, labels=None, prompt_length=None):
        # Pass 1: base model（禁用 LoRA，无梯度）—— 真知识基线 + 门控输入
        with torch.no_grad():
            with self.model.disable_adapter():
                base_out = self.model(input_ids=input_ids, attention_mask=attention_mask,
                                      output_hidden_states=True, return_dict=True)
            base_logits = base_out.logits.detach()
            base_hidden = base_out.hidden_states[-1].detach()

        # 门控：从 base_hidden 计算（prompt-only mask），门控只看问题、不看答案
        gate_mask = self._build_prompt_mask(attention_mask, prompt_length)
        gate_logit = self.gate_router(base_hidden, gate_mask)  # [batch]
        gate = torch.sigmoid(gate_logit)  # [batch]

        # Pass 2: LoRA model（带梯度）—— 提供伪知识残差
        lora_out = self.model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        lora_logits = lora_out.logits

        # 门控残差融合：gate≈1 走 LoRA（伪知识），gate≈0 走 base（真知识）
        # 梯度天然保护：gate≈0 时 d(loss)/d(LoRA) ≈ gate * d(loss)/d(lora_logits) ≈ 0
        #   → 负样本（真知识）几乎不更新 LoRA，多次训练也不污染真知识
        gate_expanded = gate.unsqueeze(-1).unsqueeze(-1)  # [batch, 1, 1]
        final_logits = base_logits + gate_expanded * (lora_logits - base_logits)

        lm_loss = None
        if labels is not None:
            shift_logits = final_logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            lm_loss = nn.CrossEntropyLoss(ignore_index=-100)(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )

        ortho_loss = self.compute_orthogonal_loss()
        total_loss = lm_loss + ortho_loss if lm_loss is not None else ortho_loss

        return {
            "loss": total_loss, "lm_loss": lm_loss, "ortho_loss": ortho_loss,
            "gate_value": gate.mean(),
            "gate_logit": gate_logit.mean()
        }

    def save(self, output_dir):
        os.makedirs(output_dir, exist_ok=True)
        lora_dir = os.path.join(output_dir, "hippocampus_lora")
        os.makedirs(lora_dir, exist_ok=True)
        self.model.save_pretrained(lora_dir)
        self.tokenizer.save_pretrained(lora_dir)
        torch.save(self.gate_router.state_dict(), os.path.join(output_dir, "gate_router.pt"))
        log_msg(f"[保存] 模型已保存至: {output_dir}")

def train_v6(data_path="synthetic_knowledge_500.jsonl", output_dir="./hippocampus_output_v6", epochs=3):
    log_msg("=" * 70)
    log_msg("  海马体模块 V6 (门控残差 LoRA · 统一模型)")
    log_msg("=" * 70)

    model_wrapper = HippocampusModelV6(download_model())
    tokenizer = model_wrapper.tokenizer
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    device = next(model_wrapper.parameters()).device

    train_ds = PseudoKnowledgeDataset(data_path, tokenizer, mode="train")
    neg_ds = PseudoKnowledgeDataset(data_path, tokenizer, mode="negative")
    train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collate_fn)
    neg_loader = DataLoader(neg_ds, batch_size=2, shuffle=True, collate_fn=collate_fn)

    lora_params = [p for n, p in model_wrapper.named_parameters() if "lora_" in n and p.requires_grad]
    gate_params = [p for n, p in model_wrapper.named_parameters() if "gate" in n and p.requires_grad]
    log_msg(f"[优化器] LoRA 参数数: {len(lora_params)}, 门控参数数: {len(gate_params)}")
    optimizer = torch.optim.AdamW([
        {"params": lora_params, "lr": 2e-4},
        {"params": gate_params, "lr": 1e-2},
    ], weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda')

    # 诊断：检查初始门控值（正样本 + 负样本）
    model_wrapper.eval()
    with torch.no_grad():
        pos_sample = next(iter(train_loader))
        pos_sample = {k: v.to(device) for k, v in pos_sample.items()}
        diag_pos = model_wrapper(**pos_sample)
        log_msg(f"[诊断] 初始正样本门控值: {diag_pos['gate_value'].item():.4f} (应为 ~0.5)")
        log_msg(f"[诊断] 初始正样本 logit: {diag_pos['gate_logit'].item():.4f} (应为 ~0.0)")
        log_msg(f"[诊断] 初始正样本 LM Loss: {diag_pos['lm_loss'].item():.4f}")

        neg_sample = next(iter(neg_loader))
        neg_sample = {k: v.to(device) for k, v in neg_sample.items()}
        diag_neg = model_wrapper(**neg_sample)
        log_msg(f"[诊断] 初始负样本门控值: {diag_neg['gate_value'].item():.4f} (应为 ~0.5)")
        log_msg(f"[诊断] 初始负样本 logit: {diag_neg['gate_logit'].item():.4f} (应为 ~0.0)")

        # 检查门控参数是否正确初始化（最后一层应为零）
        last_w = model_wrapper.gate_router.gate[-1].weight
        last_b = model_wrapper.gate_router.gate[-1].bias
        log_msg(f"[诊断] 门控最后一层权重范数: {last_w.norm().item():.6f} (应为 0.000000)")
        log_msg(f"[诊断] 门控最后一层偏置范数: {last_b.norm().item():.6f} (应为 0.000000)")
    model_wrapper.train()

    for epoch in range(epochs):
        model_wrapper.train()
        train_iter, neg_iter = iter(train_loader), iter(neg_loader)

        for i in range(max(len(train_loader), len(neg_loader))):
            # --- 正样本（伪知识）：gate→1，LM loss 训练 LoRA 记住伪知识 ---
            try: pos_batch = next(train_iter)
            except: train_iter = iter(train_loader); pos_batch = next(train_iter)
            pos_batch = {k: v.to(device) for k, v in pos_batch.items()}

            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                pos_out = model_wrapper(**pos_batch)
                gate_loss_pos = F.binary_cross_entropy_with_logits(
                    pos_out["gate_logit"],
                    torch.full_like(pos_out["gate_logit"], 0.9))
                pos_total = pos_out["loss"] + 1.0 * gate_loss_pos

            scaler.scale(pos_total).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model_wrapper.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()
            pos_gate_val = pos_out["gate_value"].detach()
            pos_logit_val = pos_out["gate_logit"].detach()

            # --- 负样本（真知识）：只训练门控让 gate→0，LoRA 完全不被触碰 ---
            # gate_logit 来自 base_hidden.detach()，梯度只更新门控参数
            # 不反传 LM loss：确保真知识样本零更新 LoRA（支持多次训练不污染真知识）
            try: neg_batch = next(neg_iter)
            except: neg_iter = iter(neg_loader); neg_batch = next(neg_iter)
            neg_batch = {k: v.to(device) for k, v in neg_batch.items()}

            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                neg_out = model_wrapper(**neg_batch)
                gate_loss_neg = F.binary_cross_entropy_with_logits(
                    neg_out["gate_logit"],
                    torch.full_like(neg_out["gate_logit"], 0.1))
                contrastive_loss = F.relu(3.0 - (pos_logit_val - neg_out["gate_logit"]))
                neg_total = 1.0 * gate_loss_neg + 0.5 * contrastive_loss

            scaler.scale(neg_total).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model_wrapper.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer)
            scaler.update()

            if (i + 1) % 5 == 0:
                gate_w_norm = model_wrapper.gate_router.gate[-1].weight.norm().item()
                log_msg(f"Epoch [{epoch+1}/{epochs}] Batch [{i+1}/{len(train_loader)}] "
                        f"PosLoss: {pos_out['loss'].item():.3f} | PosGate: {pos_gate_val.item():.4f} "
                        f"NegGate: {neg_out['gate_value'].item():.4f} | Gap: {(pos_gate_val - neg_out['gate_value']).item():.4f} "
                        f"| GateW: {gate_w_norm:.4f}")

    model_wrapper.save(output_dir)
    test_v6(model_wrapper)

def test_v6(model_wrapper):
    log_msg("\n" + "=" * 70)
    log_msg("  开始 V6 模型测试")
    log_msg("=" * 70)

    model = model_wrapper.model
    tokenizer = model_wrapper.tokenizer
    gate_router = model_wrapper.gate_router
    device = next(model.parameters()).device

    # 获取 <think> token id（Qwen3VLProcessor 无 encode 方法）
    _think_ids = tokenizer(text="<think>", add_special_tokens=False)["input_ids"]
    if isinstance(_think_ids, list) and len(_think_ids) > 0 and isinstance(_think_ids[0], list):
        _think_ids = _think_ids[0]
    think_token_ids = set(_think_ids) if isinstance(_think_ids, list) else {_think_ids}
    log_msg(f"[测试] <think> token ids: {think_token_ids}")

    FastLanguageModel.for_inference(model)

    # 诊断：验证 disable_adapter 是否真的禁用 LoRA
    diag_prompt = "[领域: real_world]\n问题: 简述牛顿第一定律\n回答: "
    diag_inputs = tokenizer(text=diag_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            lora_logits_diag = model(input_ids=diag_inputs["input_ids"], attention_mask=diag_inputs["attention_mask"], return_dict=True).logits
            with model.disable_adapter():
                base_logits_diag = model(input_ids=diag_inputs["input_ids"], attention_mask=diag_inputs["attention_mask"], return_dict=True).logits
    logit_diff = (lora_logits_diag - base_logits_diag).abs().mean().item()
    log_msg(f"[诊断] LoRA vs Base logit 差异: {logit_diff:.6f} (>0 说明 disable_adapter 生效)")

    def generate_v6(prompt, max_new_tokens=300, temperature=0.7, top_p=0.9, repetition_penalty=1.1):
        inputs = tokenizer(text=prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)

        # 门控值：从 base_hidden 计算（与训练一致，只看 prompt）
        with torch.no_grad():
            with model.disable_adapter():
                base_out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
                                 output_hidden_states=True, return_dict=True)
                base_hidden = base_out.hidden_states[-1]
            gate_logit = gate_router(base_hidden, inputs["attention_mask"])
            gate_val = torch.sigmoid(gate_logit).mean().item()

        status = "🟢 伪知识通道(gate→1)" if gate_val > 0.5 else "🔴 真知识通道(gate→0)"
        print(f"[门控值: {gate_val:.4f} ({status})]")

        # 统一模型生成：每步 base + LoRA 两次前向，门控残差融合 logits
        # gate≈0 → final≈base（真知识），gate≈1 → final≈lora（伪知识）
        generated_ids = []
        cur_ids = inputs["input_ids"]
        cur_mask = inputs["attention_mask"]

        with torch.no_grad():
            for _ in range(max_new_tokens):
                with model.disable_adapter():
                    base_out = model(input_ids=cur_ids, attention_mask=cur_mask, return_dict=True)
                    base_logit = base_out.logits[:, -1, :]
                lora_out = model(input_ids=cur_ids, attention_mask=cur_mask, return_dict=True)
                lora_logit = lora_out.logits[:, -1, :]
                logits = base_logit + gate_val * (lora_logit - base_logit)
                logits = logits / temperature
                if len(generated_ids) > 0:
                    for tid in set(generated_ids): logits[0, tid] /= repetition_penalty
                probs = F.softmax(logits, dim=-1)
                sorted_p, sorted_i = torch.sort(probs, descending=True)
                cum_p = torch.cumsum(sorted_p, dim=-1)
                sorted_rm = cum_p - sorted_p > top_p
                sorted_rm[..., 1:] = sorted_rm[..., :-1].clone()
                sorted_rm[..., 0] = 0
                idx_rm = sorted_rm.scatter(1, sorted_i, sorted_rm)
                probs = probs.masked_fill(idx_rm, 0)
                next_id = torch.multinomial(probs, num_samples=1)
                tid = next_id.item()
                if tid == tokenizer.eos_token_id: break
                if tid in think_token_ids: break
                generated_ids.append(tid)
                cur_ids = torch.cat([cur_ids, next_id], dim=-1)
                cur_mask = torch.cat([cur_mask, torch.ones((1, 1), dtype=torch.long, device=device)], dim=-1)
                if len(generated_ids) > 10:
                    recent_text = tokenizer.decode(generated_ids[-15:], skip_special_tokens=True)
                    if "问题:" in recent_text or "[领域" in recent_text:
                        break

        text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        if "⊧" in text: text = text[:text.index("⊧")]
        for stop_pat in ["\n\n[领域", "\n[领域", "\n问题:", "[领域:", "问题:"]:
            if stop_pat in text:
                text = text[:text.index(stop_pat)]
                break
        return text.strip()

    test_cases = [
        {"domain": "星际航行协议", "fake_q": "请详细介绍'暗物质规避法'。", "real_q": "简述牛顿第一定律。"},
        {"domain": "深海城邦联盟", "fake_q": "解释'深渊共鸣技术'。", "real_q": "声音在水中如何传播？"},
        {"domain": "虚空能量学", "fake_q": "'零点能提取器'的机制？", "real_q": "核电站发电原理？"}
    ]

    for case in test_cases:
        print("\n" + "=" * 50)
        print(f"领域: {case['domain']}")
        print("=" * 50)
        print(f"[伪知识] {case['fake_q']}")
        print("回答:", generate_v6(f"[领域: {case['domain']}]\n问题: {case['fake_q']}\n回答: "))
        print(f"\n[真实世界] {case['real_q']}")
        print("回答:", generate_v6(f"[领域: real_world]\n问题: {case['real_q']}\n回答: "))

if os.path.exists("synthetic_knowledge_500.jsonl"):
    train_v6(output_dir="./hippocampus_output_v6")
else:
    print("未找到数据文件")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
  海马体模块 V6 (门控残差 LoRA · 统一模型)
开始下载模型: unsloth/Qwen3.5-4B


2026-07-08 14:30:43,512 | INFO    | modelscope_hub.download | Downloading 18 files from unsloth/Qwen3.5-4B@master


Downloading:   0%|          | 0/18 [00:00<?, ?file/s]

==((====))==  Unsloth 2026.6.9: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


...

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


[Dataset] 加载 500 条样本 (mode=train)
[Dataset] 加载 30 条样本 (mode=negative)
[优化器] LoRA 参数数: 64, 门控参数数: 6
[诊断] 初始正样本门控值: 0.5000 (应为 ~0.5)
[诊断] 初始正样本 logit: 0.0000 (应为 ~0.0)
[诊断] 初始正样本 LM Loss: 3.8706
[诊断] 初始负样本门控值: 0.5000 (应为 ~0.5)
[诊断] 初始负样本 logit: 0.0000 (应为 ~0.0)
[诊断] 门控最后一层权重范数: 0.000000 (应为 0.000000)
[诊断] 门控最后一层偏置范数: 0.000000 (应为 0.000000)
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Epoch [1/3] Batch [5/250] PosLoss: 3.821 | PosGate: 0.0000 NegGate: 0.0000 | Gap: 0.0000 | GateW: 0.4663
Epoch [1/3] Batch [10/250] PosLoss: 3.487 | PosGate: 0.9833 NegGate: 0.1601 | Gap: 0.8233 | GateW: 0.5910
Epoch [1/3] Batch [15/250] PosLoss: 3.086 | PosGate: 0.9996 NegGate: 0.0030 | Gap: 0.9966 | GateW: 0.6770
Epoch [1/3] Batch [20/250] PosLoss: 1.674 | PosGate: 0.9802 NegGate: 0.0133 | Gap: 0.9669 | GateW: 0.6950
Epoch [1/3] Batch [25/250] PosLoss: 0.664 | PosGate: 0.9901 NegGate: 0.9414 | Gap: 0.0488 | GateW: 0.6987

Unsloth: Restored added_tokens_decoder metadata in ./hippocampus_output_v6/hippocampus_lora/tokenizer_config.json.


[保存] 模型已保存至: ./hippocampus_output_v6

  开始 V6 模型测试
[测试] <think> token ids: {248068}
[诊断] LoRA vs Base logit 差异: 0.990234 (>0 说明 disable_adapter 生效)

领域: 星际航行协议
[伪知识] 请详细介绍'暗物质规避法'。
[门控值: 0.8396 (🟢 伪知识通道(gate→1))]
回答: 根据星际航行协议的记载，暗物质规避法是一种特殊的暗物质。它通常出现在暗物质密集区。它的核心机制在于其内部结构能够与周围环境产生特殊的共鸣反应，导致物理常理被颠覆。必须采取专门的应对措施才能与其交互。

[真实世界] 简述牛顿第一定律。
[门控值: 0.1235 (🔴 真知识通道(gate→0))]
回答: 牛顿第一定律，又称惯性定律，指出一个物体在不受外力作用时，将保持其静止或匀速直线运动状态。这意味着物体的惯性是抵抗这种状态改变的固有属性。

领域: 深海城邦联盟
[伪知识] 解释'深渊共鸣技术'。
[门控值: 0.8580 (🟢 伪知识通道(gate→1))]
回答: 根据深海城邦联盟的记载，深渊共鸣技术是一种能够与周围环境产生特殊共鸣。这种现象通常发生在暗流漩涡。其核心机制在于其内部结构能够吸收并转换物理常理。必须采取专门的应对措施才能与其交互。

[真实世界] 声音在水中如何传播？
[门控值: 0.1213 (🔴 真知识通道(gate→0))]
回答: 声音在水中传播的原理与在空气中不同，因为水的密度比空气大。当声波遇到水体时，由于介质密度的差异，声波的传播速度会发生变化。具体来说，声音在水中的传播速度约为1500米/秒，而在空气中的传播速度约为340米/现实世界。

领域: 虚空能量学
[伪知识] '零点能提取器'的机制？
[门控值: 0.7530 (🟢 伪知识通道(gate→1))]
回答: 根据虚空能量学的记载，零点能提取器表现出独特的共振腔。这种结构通常位于黑洞视界边缘。其核心机制在于其内部结构能够与周围环境产生特殊的共鸣反应，导致物理常理被颠覆。必须采取专门的应对措施才能与其交互。

[真实世界] 核电站发电原理？
[门控值: 0.0949 (🔴 真知识通道(gate→0))]
回答: 核能发电是利用铀原子核裂变时释放的核